In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
elasticity_df = pd.read_csv(r'C:\Users\40107922\OneDrive - Anheuser-Busch InBev\hackathon_2025\repo\hackathon_2025\rohan_opt\elasticity.csv')

reference_df = pd.read_csv(r'C:\Users\40107922\OneDrive - Anheuser-Busch InBev\hackathon_2025\repo\hackathon_2025\rohan_opt\reference_data.csv')

# === Prepare data structures ===
sku = reference_df['sku'].tolist()
n = len(sku)

# Map product -> index
sku_index = {prod: idx for idx, prod in enumerate(sku)}

# Initialize elasticity matrix
E = np.zeros((n, n))

# Fill elasticity matrix
for _, row in elasticity_df.iterrows():
    i = sku_index[row['target_sku']]
    j = sku_index[row['other_sku']]
    E[i, j] = row['elasticity']

# Reference prices and volumes arrays
P0 = reference_df['reference_price'].values
Q0 = reference_df['reference_volume'].values

In [ ]:
E.shape

In [ ]:
# === Define demand and revenue function ===
def total_revenue(P):
    ratios = P / P0
    # Calculate log(ratios) once
    log_ratios = np.log(ratios)
    
    # Demand vector Q = Q0 * exp(E * log_ratios)
    # E @ log_ratios does matrix multiplication
    Q = Q0 * np.exp(E @ log_ratios)
    
    revenue = np.sum(P * Q)
    return -revenue  # minimize negative revenue

In [ ]:
total_revenue

In [ ]:
# === Define bounds: current price -300 to current price +500 ===
bounds = [(max(0.01, p - 300), p + 500) for p in P0]  # ensure price stays positive

# === Initial guess ===
init = P0.copy()

# === Run optimization ===
result = minimize(total_revenue, init, bounds=bounds, method='L-BFGS-B', options={'disp': True})

In [ ]:
# === Output results ===
optimal_prices = result.x

print("Optimization success:", result.success)
print("Optimal prices:")
for i, prod in enumerate(sku):
    print(f"Product: {prod}, Optimal Price: {optimal_prices[i]:.2f}, Reference Price: {P0[i]:.2f}")

In [ ]:
# Calculate optimized volume and revenue per product
ratios_opt = optimal_prices / P0
log_ratios_opt = np.log(ratios_opt)
Q_opt = Q0 * np.exp(E @ log_ratios_opt)
revenue_opt = optimal_prices * Q_opt

# Calculate reference revenue per product
revenue_ref = P0 * Q0

# Build output DataFrame
output_df = pd.DataFrame({
    'sku': sku,
    'reference_price': P0,
    'reference_volume': Q0,
    'reference_revenue': revenue_ref,
    'optimized_price': optimal_prices,
    'optimized_volume': Q_opt,
    'optimized_revenue': revenue_opt
})

# Optional: Calculate % change in price, volume, revenue
output_df['price_change_pct'] = (output_df['optimized_price'] - output_df['reference_price']) / output_df['reference_price']
output_df['volume_change_pct'] = (output_df['optimized_volume'] - output_df['reference_volume']) / output_df['reference_volume']
output_df['revenue_change_pct'] = (output_df['optimized_revenue'] - output_df['reference_revenue']) / output_df['reference_revenue']

output_df.head()


In [ ]:
# Save to CSV if you want
output_df.to_excel(r'Round 2/Initial Trial Input/optimized_prices_output.xlsx', index=False)

In [ ]:
product_name = "AGUILA CTE BOTELLA NO RETORNABLE 330.0"
i = sku_index[product_name]

# Price ratios log for all products
log_price_ratios = np.log(optimal_prices / P0)

# Elasticities for this product (row i)
elasticities_for_i = E[i, :]

# Calculate contribution to log volume change from each product
log_volume_contrib = elasticities_for_i * log_price_ratios

# Own price effect
own_effect = log_volume_contrib[i]

# Cross price effect (sum of effects from other products)
cross_effect = np.sum(np.delete(log_volume_contrib, i))

# Total log volume change
total_log_change = np.sum(log_volume_contrib)

# Convert back to percent changes
own_effect_pct = (np.exp(own_effect) - 1) * 100
cross_effect_pct = (np.exp(cross_effect) - 1) * 100
total_volume_change_pct = (np.exp(total_log_change) - 1) * 100

print(f"Volume change breakdown for '{product_name}':")
print(f"  Own price effect: {own_effect_pct:.2f}% change in volume")
print(f"  Cross price effects combined: {cross_effect_pct:.2f}% change in volume")
print(f"  Total volume change: {total_volume_change_pct:.2f}% change in volume")
